# Day 21: GPU Online Attention 与 Fused SwiGLU

Triton kernel 对 Q/K/V 分块，在线维护 m/l/o，不写出完整 score。它仅支持等长 causal prefill、D=16/32/64、forward，无 backward/GQA/padding。

参考源码：`learning/kernels/triton_ops.py`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY15_21.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 21完整讲义](../docs/DAY21_FUSED_KERNELS.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

像加工工序直接传给下一道，不把每个半成品送回大仓库：

```text
Attention：Q/K局部分数 → m/l/o更新 → 下一块 → 最终O
SwiGLU：  gate → SiLU ─┐
          up ──────────┴→ 相乘 → 输出
```

gate=2、up=3时，输出为 `SiLU(2)*3 ≈ 5.2848`。
这里的SiLU可以输出负数或大于1，不是路由概率。
融合SwiGLU本例只融合SiLU与乘法，不包含E→F的两次投影及F→E的GEMM。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(21, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

为何能称教学融合 Attention，不能称完整生产 FlashAttention？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：为什么融合SwiGLU不等于融合整个FFN？**
答：三个线性投影GEMM不在该逐元素kernel里，它只接收已算出的gate/up。

**问：融合越多越快吗？**
答：不一定，资源占用与调度自由度下降可能超过减少IO/launch的收益。

**问：优化kernel最少交付什么？**
答：支持契约、独立正确性、负例/边界、精度、同步计时、环境与适用范围。

**问：m/l/o能替代KV Cache吗？**
答：不能，m/l/o是当前Query的累积状态，KV是供不同Query读取的历史表示。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY21_FUSED_KERNELS.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。